# テキスト分類器をゼロから作る

**実践編（総合演習）／ Building a Text Classifier from Scratch**

公開コーパス **20 Newsgroups** を題材に、前処理 → データ分割 → TF-IDF → 分類（ナイーブベイズ vs ロジスティック回帰）→ 評価（適合率・再現率・F値・混同行列・交差検証）までを一続きに作ります。

このノートブックは学習ページ *practice-text-classifier.html* のコードと同じ内容です。Colab では上から順に実行できます（`fetch_20newsgroups` は初回にデータをネットワーク経由で取得します）。


## 準備：ライブラリのインストール

Colab には scikit-learn が最初から入っていますが、念のため明示します。


In [ ]:
!pip install -q scikit-learn numpy

## Step 1. データを手に入れる

話題の異なる4グループに絞り、本文以外の手がかり（ヘッダ・署名・引用）は取り除きます。


In [ ]:
from sklearn.datasets import fetch_20newsgroups

# 話題の異なる4グループだけを対象にする
categories = [
    "alt.atheism",
    "comp.graphics",
    "sci.med",
    "soc.religion.christian",
]

# 本文以外の手がかり（ヘッダ・署名・引用）は取り除く
data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=42,
)

print("文書数:", len(data.data))
print("クラス名:", data.target_names)

# グループごとの件数を数える
import numpy as np
for i, name in enumerate(data.target_names):
    print(name, (data.target == i).sum())

In [ ]:
i = 0
print("ラベル:", data.target[i], "->", data.target_names[data.target[i]])
print("本文の冒頭:")
print(data.data[i][:200])

## Step 2. 前処理とデータ分割

まず前処理の考え方を1文で確認し、次に `train_test_split` で学習用と評価用に分けます。`stratify` でグループの割合を保ちます。


In [ ]:
import re

# 前処理の考え方を1文で確認する（実際の学習では Vectorizer に任せる）
sample = data.data[0]
stop = {"the", "is", "it", "as", "a", "an", "of", "to", "and"}

tokens = re.findall(r"[a-z]+", sample.lower())   # 小文字化してトークン化
tokens = [t for t in tokens if t not in stop]    # ストップワード除去
print(tokens[:12])

In [ ]:
from sklearn.model_selection import train_test_split

X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,       # グループの割合を保ったまま分割
    random_state=42,
)
print("学習用:", len(X_train), "評価用:", len(X_test))

## Step 3. 特徴量にする（TF-IDF）

`TfidfVectorizer` でテキストを数値ベクトルに変えます。idf は scikit-learn の既定（平滑化）で `ln((1+N)/(1+df)) + 1`、各文書は L2 正規化されます。


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",   # ありふれた語を除く
    min_df=5,               # 5文書未満の語は捨てる
    sublinear_tf=True,      # tf を 1 + log(tf) に圧縮
)

# 学習用だけで語彙とIDFを学習し、そのまま変換する
X_train_vec = vectorizer.fit_transform(X_train)
print("行列の形:", X_train_vec.shape)
print("語彙の例:", vectorizer.get_feature_names_out()[500:506])

## Step 4. 分類器を学習する

ベクトル化と分類器を `Pipeline` でつなぎ、リークを防ぎます。ナイーブベイズとロジスティック回帰を同じ条件で学習します。


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

def make_vectorizer():
    return TfidfVectorizer(stop_words="english", min_df=5, sublinear_tf=True)

# 2つのパイプライン（前処理 + 分類器）
nb = Pipeline([
    ("tfidf", make_vectorizer()),
    ("clf", MultinomialNB()),
])
lr = Pipeline([
    ("tfidf", make_vectorizer()),
    ("clf", LogisticRegression(max_iter=1000, C=10)),
])

# 学習用データだけで学習する（前処理も含めて fit される）
nb.fit(X_train, y_train)
lr.fit(X_train, y_train)
print("学習が終わりました")

In [ ]:
examples = [
    "The patient was prescribed antibiotics for the infection.",
    "I rendered the 3D scene with ray tracing and saved the image.",
]
for text, pred in zip(examples, lr.predict(examples)):
    print(data.target_names[pred], "<-", text)

## Step 5. 評価する

精度だけでなく、クラスごとの適合率・再現率・F値（`classification_report`）、間違いの内訳（`confusion_matrix`）、ばらつき（`cross_val_score`）を見ます。


In [ ]:
from sklearn.metrics import classification_report

y_pred_nb = nb.predict(X_test)
print(classification_report(
    y_test, y_pred_nb, target_names=data.target_names, digits=2))

In [ ]:
y_pred_lr = lr.predict(X_test)
print(classification_report(
    y_test, y_pred_lr, target_names=data.target_names, digits=2))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_nb)
print(data.target_names)
print(cm)

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in [("NB", nb), ("LR", lr)]:
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name}: 平均 {scores.mean():.3f} (+/- {scores.std():.3f})")
    print("  各分割:", [round(s, 3) for s in scores])

## Step 6. 結果を読み解く

ロジスティック回帰の係数から、各クラスで手がかりになっている語を確かめます。線形モデルの解釈しやすさは長所です。


In [ ]:
import numpy as np

vec = lr.named_steps["tfidf"]
clf = lr.named_steps["clf"]
feature_names = np.array(vec.get_feature_names_out())

for i, name in enumerate(data.target_names):
    top = feature_names[np.argsort(clf.coef_[i])[-6:]][::-1]
    print(f"{name}: {list(top)}")

## 発展課題

- `categories` に別グループ（例：`rec.sport.baseball`, `sci.space`）を足してクラス数を増やす
- `TfidfVectorizer` の `ngram_range=(1, 2)` や `min_df` / `max_df` を変える
- 別コーパス（日本語なら livedoor ニュースコーパス。分かち書きを挟む）で同じ流れを試す
- ロジスティック回帰の係数が負に大きい語や、誤分類された記事を1件ずつ読む
